# Quick Sort and Partition Mechanics

**Session:** 2026-11-08 · Dynamic Programming and Divide-and-Conquer  
**Module:** 01 · Python Fundamentals and Algorithmic Thinking  

---

## 1. Quick Sort Strategy: In-Place Partitioning

Like Merge Sort, **Quick Sort** follows the Divide-and-Conquer paradigm. However, the computational work is concentrated in the **Divide** step rather than the **Combine** step:
- **Divide:** Select a *pivot* element $x \in S$. Partition array $S$ into subarrays $L$ (elements $\le x$) and $R$ (elements $> x$).
- **Conquer:** Recursively sort subarrays $L$ and $R$.
- **Combine:** Trivial! Because all elements in $L$ precede the pivot and all elements in $R$ succeed the pivot, no concatenation work is required.


In [ ]:
from __future__ import annotations

import json
import random
from pathlib import Path

DATA_DIR = Path("../data")
with (DATA_DIR / "sorting_benchmarks.json").open(encoding="utf-8") as f:
    benchmarks = json.load(f)

## 2. Lomuto Partition Scheme

In Lomuto's partition scheme:
1. The rightmost element `S[high]` is chosen as the pivot.
2. A scan pointer `i` scans from `low` to `high - 1`.
3. A store pointer `pointer` tracks the boundary of elements $\le \text{pivot}$.
4. When `S[i] <= pivot`, swap `S[i]` with `S[pointer]` and advance `pointer`.
5. Finally, swap `S[high]` with `S[pointer]`. The pivot is now at index `pointer`.


In [ ]:
def partition_lomuto(low: int, high: int, s: list[int]) -> int:
    """Lomuto partition scheme using s[high] as pivot."""
    pivot = s[high]
    pointer = low

    for i in range(low, high):
        if s[i] <= pivot:
            s[i], s[pointer] = s[pointer], s[i]
            pointer += 1

    s[pointer], s[high] = s[high], s[pointer]
    return pointer


def quicksort_lomuto(low: int, high: int, s: list[int]) -> None:
    """Quick Sort using Lomuto partitioning."""
    if low < high:
        p = partition_lomuto(low, high, s)
        quicksort_lomuto(low, p - 1, s)
        quicksort_lomuto(p + 1, high, s)


# Test on lecture array
arr = benchmarks["lecture_example"]["array"].copy()
print("Before Lomuto Quick Sort:", arr)
quicksort_lomuto(0, len(arr) - 1, arr)
print("After Lomuto Quick Sort: ", arr)
assert arr == sorted(benchmarks["lecture_example"]["array"])

## 3. Hoare Partition Scheme

C.A.R. Hoare's original partition scheme uses two pointers converging from both ends of the subarray toward each other. It performs roughly three times fewer swaps on average than Lomuto's scheme.


In [ ]:
def partition_hoare(low: int, high: int, s: list[int]) -> int:
    """Hoare partition scheme with convergent two-pointer scans."""
    pivot = s[(low + high) // 2]
    i = low - 1
    j = high + 1

    while True:
        i += 1
        while s[i] < pivot:
            i += 1

        j -= 1
        while s[j] > pivot:
            j -= 1

        if i >= j:
            return j

        s[i], s[j] = s[j], s[i]


def quicksort_hoare(low: int, high: int, s: list[int]) -> None:
    """Quick Sort using Hoare partitioning."""
    if low < high:
        p = partition_hoare(low, high, s)
        quicksort_hoare(low, p, s)
        quicksort_hoare(p + 1, high, s)


arr = benchmarks["lecture_example"]["array"].copy()
quicksort_hoare(0, len(arr) - 1, arr)
assert arr == sorted(benchmarks["lecture_example"]["array"])
print("Hoare Quick Sort verified successfully on lecture array.")

## 4. Randomized Quick Sort: Eliminating $O(n^2)$ Worst-Case

When the pivot is deterministically chosen as the first or last element, already sorted or reverse-sorted sequences produce maximally unbalanced partitions ($0$ and $n-1$), degrading performance to $O(n^2)$.

By randomly selecting a pivot element and swapping it with `s[high]` before partitioning, the expected running time is guaranteed to be $O(n \log n)$ for all inputs.


In [ ]:
def partition_randomized(low: int, high: int, s: list[int]) -> int:
    """Randomized pivot selection to guarantee expected O(n log n) runtime."""
    pivot_idx = random.randint(low, high)
    s[pivot_idx], s[high] = s[high], s[pivot_idx]
    return partition_lomuto(low, high, s)


def quicksort_randomized(low: int, high: int, s: list[int]) -> None:
    """Randomized Quick Sort."""
    if low < high:
        p = partition_randomized(low, high, s)
        quicksort_randomized(low, p - 1, s)
        quicksort_randomized(p + 1, high, s)


# Test on reverse sorted benchmark (pathological for deterministic Lomuto)
reverse_case = benchmarks["reverse_sorted"]["array"].copy()
quicksort_randomized(0, len(reverse_case) - 1, reverse_case)
assert reverse_case == sorted(benchmarks["reverse_sorted"]["array"])
print("Randomized Quick Sort successfully sorted pathological reverse array:", reverse_case)

## 5. Comprehensive Test Across All Benchmarks


In [ ]:
for name, case in benchmarks.items():
    arr = case["array"].copy()
    if len(arr) > 0:
        quicksort_randomized(0, len(arr) - 1, arr)
    assert arr == sorted(case["array"])
    print(f"PASS: {name}")